# Classificadores para Credit Data

Notebook canônico para comparar regressão logística, Naive Bayes, random forest e SVM. O conjunto de teste preservado no artefato legado permanece isolado até a avaliação final. O SVM é selecionado por validação cruzada somente no treino.

> `pickle` pode executar código. Este notebook aceita apenas o artefato versionado neste repositório privado; não substitua o arquivo por um download não verificado.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from ml_studies.loaders import load_trusted_legacy_pickle

In [ ]:
candidates = [Path('Parte2/credit.pkl'), Path('3_NaiveBayes/Parte2/credit.pkl')]
legacy_path = next((path for path in candidates if path.is_file()), None)
if legacy_path is None:
    raise FileNotFoundError('credit.pkl canônico não encontrado')
X_train, y_train, X_test, y_test = load_trusted_legacy_pickle(
    legacy_path, trusted=True
)[:4]
len(X_train), len(X_test)

In [ ]:
models = {
    'regressao_logistica': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(random_state=42)),
    ]),
    'naive_bayes': Pipeline([
        ('scaler', StandardScaler()),
        ('model', GaussianNB()),
    ]),
    'random_forest': RandomForestClassifier(
        n_estimators=100, class_weight='balanced', random_state=42
    ),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    rows.append({
        'modelo': name,
        'acuracia': accuracy_score(y_test, predictions),
        'macro_f1': f1_score(y_test, predictions, average='macro', zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_test, predictions),
    })

In [ ]:
svm = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('model', SVC())]),
    {'model__kernel': ['linear', 'rbf', 'poly'], 'model__C': [0.1, 1, 10]},
    scoring='f1_macro',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=1,
)
svm.fit(X_train, y_train)
predictions = svm.predict(X_test)
rows.append({
    'modelo': 'svm',
    'acuracia': accuracy_score(y_test, predictions),
    'macro_f1': f1_score(y_test, predictions, average='macro', zero_division=0),
    'balanced_accuracy': balanced_accuracy_score(y_test, predictions),
})
results = pd.DataFrame(rows).set_index('modelo').sort_values('macro_f1', ascending=False)
print('SVM selecionado no treino:', svm.best_params_)
results